In [ ]:
# =============================================================================
# Experiment 12: Context-Based Malicious Event Detection in Wireless Sensor Networks (WSN)
# Objective: Develop a model that uses context (e.g., timestamp, source IP, and event type) to detect malicious events within wireless sensor networks. [CO5]
# =============================================================================

# 1. Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =============================================================================
# 2. Generate Synthetic Wireless Sensor Network (WSN) Contextual Telemetry Dataset
# Simulating sensor node communications with Timestamp, Source IP, Event Type, Packet Size, and Signal Strength.
# Label: 0 = Normal Sensor Event, 1 = Malicious WSN Event (e.g., Sinkhole, Sybil, Spoofed Node, Off-Hours Exfiltration)
# =============================================================================

np.random.seed(42)
n_normal = 800
n_malicious = 200

# Normal WSN node behavior (authorized internal sensor subnet, daytime/scheduled intervals, routine telemetry)
normal_ips = ['10.0.1.11', '10.0.1.12', '10.0.1.13', '10.0.1.14', '10.0.1.15', '10.0.1.16']
normal_events = ['TEMP_READING', 'HUMIDITY_READING', 'HEARTBEAT', 'BATTERY_STATUS']

normal_hours = np.random.choice(range(6, 22), size=n_normal)
normal_timestamps = [
    pd.Timestamp('2026-11-14') + pd.Timedelta(hours=int(h), minutes=int(np.random.randint(0, 60)))
    for h in normal_hours
]

normal_df = pd.DataFrame({
    'timestamp': normal_timestamps,
    'source_ip': np.random.choice(normal_ips, size=n_normal),
    'event_type': np.random.choice(normal_events, size=n_normal, p=[0.4, 0.3, 0.2, 0.1]),
    'payload_bytes': np.random.normal(64, 10, size=n_normal).astype(int),
    'rssi_dbm': np.random.normal(-55, 5, size=n_normal).round(1),
    'label': 0
})

# Malicious WSN behavior (rogue/spoofed IPs, late-night odd hours, unauthorized routing/config events, large payloads)
malicious_ips = ['10.0.1.99', '192.168.50.23', '10.0.1.11', '172.16.0.44']  # Includes rogue IPs & compromised node 10.0.1.11
malicious_events = ['ROUTE_OVERRIDE', 'FIRMWARE_PUSH', 'NODE_CLONE_JOIN', 'BULK_DATA_DUMP']

malicious_hours = np.random.choice([0, 1, 2, 3, 4, 23], size=n_malicious)
malicious_timestamps = [
    pd.Timestamp('2026-11-14') + pd.Timedelta(hours=int(h), minutes=int(np.random.randint(0, 60)))
    for h in malicious_hours
]

malicious_df = pd.DataFrame({
    'timestamp': malicious_timestamps,
    'source_ip': np.random.choice(malicious_ips, size=n_malicious),
    'event_type': np.random.choice(malicious_events, size=n_malicious),
    'payload_bytes': np.random.normal(512, 90, size=n_malicious).astype(int),
    'rssi_dbm': np.random.normal(-82, 8, size=n_malicious).round(1),
    'label': 1
})

# Combine and shuffle dataset
df = pd.concat([normal_df, malicious_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total WSN Events: {len(df)} | Normal (0): {(df['label']==0).sum()} | Malicious (1): {(df['label']==1).sum()}")
print(df.head())

In [ ]:
# =============================================================================
# 3. Contextual Feature Engineering (Temporal, Network & Event Context)
# Extracting hour-of-day, off-hours indicator, authorized subnet check, and event context.
# =============================================================================

authorized_nodes = {'10.0.1.11', '10.0.1.12', '10.0.1.13', '10.0.1.14', '10.0.1.15', '10.0.1.16'}

def extract_wsn_context(data):
    ctx = pd.DataFrame()
    ctx['hour_of_day'] = data['timestamp'].dt.hour
    ctx['is_off_hours'] = ((ctx['hour_of_day'] < 6) | (ctx['hour_of_day'] >= 22)).astype(int)
    ctx['is_authorized_ip'] = data['source_ip'].isin(authorized_nodes).astype(int)
    ctx['source_ip'] = data['source_ip']
    ctx['event_type'] = data['event_type']
    ctx['payload_bytes'] = data['payload_bytes']
    ctx['rssi_dbm'] = data['rssi_dbm']
    return ctx

X = extract_wsn_context(df)
y = df['label']

print("--- Extracted Contextual Features Preview ---")
print(X.head())

In [ ]:
# =============================================================================
# 4. Build Preprocessing Pipeline & Train Context-Aware Classifier
# =============================================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

categorical_cols = ['source_ip', 'event_type']
numeric_cols = ['hour_of_day', 'is_off_hours', 'is_authorized_ip', 'payload_bytes', 'rssi_dbm']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numeric_cols)
    ]
)

wsn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

wsn_pipeline.fit(X_train, y_train)
y_pred = wsn_pipeline.predict(X_test)

print(f"WSN Context Detection Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Normal Event', 'Malicious Event']))

In [ ]:
# =============================================================================
# 5. Visualize Confusion Matrix, Temporal Context & Live Event Testing
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='YlGnBu',
            xticklabels=['Normal', 'Malicious'], yticklabels=['Normal', 'Malicious'], ax=axes[0])
axes[0].set_title('WSN Context-Based Detection Confusion Matrix')
axes[0].set_xlabel('Predicted Event Status')
axes[0].set_ylabel('Actual Event Status')

# Event Distribution by Hour of Day
df['Hour'] = df['timestamp'].dt.hour
sns.histplot(data=df, x='Hour', hue='label', bins=24, multiple='stack',
             palette={0: 'steelblue', 1: 'crimson'}, ax=axes[1])
axes[1].set_title('WSN Events by Hour of Day (Temporal Context)')
axes[1].set_xlabel('Hour of Day (0 - 23)')
axes[1].set_ylabel('Event Count')
axes[1].legend(title='Event Class', labels=['Malicious (1)', 'Normal (0)'])

plt.tight_layout()
plt.show()

# Test live contextual WSN events
new_events = pd.DataFrame([
    {'timestamp': pd.Timestamp('2026-11-14 14:20:00'), 'source_ip': '10.0.1.12', 'event_type': 'TEMP_READING', 'payload_bytes': 60, 'rssi_dbm': -54.0},
    {'timestamp': pd.Timestamp('2026-11-14 02:45:00'), 'source_ip': '10.0.1.11', 'event_type': 'ROUTE_OVERRIDE', 'payload_bytes': 490, 'rssi_dbm': -80.5}
])

new_preds = wsn_pipeline.predict(extract_wsn_context(new_events))

print("--- Live Contextual WSN Event Inspection ---")
for (_, row), pred in zip(new_events.iterrows(), new_preds):
    status = 'MALICIOUS EVENT' if pred == 1 else 'NORMAL EVENT'
    print(f"[{status}] Time: {row['timestamp']} | IP: {row['source_ip']} | Type: {row['event_type']}")